In [1]:
!pip install fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.4 MB/s eta 0:00:00


In [2]:
import os
import sklearn
import pandas as pd
from joblib import load
import json

In [3]:
!ngrok authtoken 2tQR3hMKgY1918rkKxTdGfu9g5s_4S59sDie9wVDY55tEjvZa

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [9]:
%%writefile app.py

import os
import sklearn
import pandas as pd
import numpy as np
from joblib import load
from fastapi import FastAPI, HTTPException
from contextlib import asynccontextmanager
from pydantic import BaseModel

app = FastAPI()

# Define input data schema
class PredictionRequest(BaseModel):
    Age: int
    KM_Driven: int
    Fuel_Type: str
    Transmission: str
    Owner_Type: str
    Model: str

ml_model = load('cars_maruti.pkl')

# Prediction endpoint
@app.post("/predict")
def predict(input_data: PredictionRequest):
    try:
        # Convert input data to a dictionary for prediction
        input_dict = input_data.dict()

        df = pd.DataFrame(input_dict, index = [0])

        # Call the model's prediction method
        prediction = ml_model.predict(df)

        # Return the prediction result
        return {f"Estimated Car Price: INR {np.round(prediction[0], 2)} Lakhs"}

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {e}")


Overwriting app.py


In [10]:
!nohup uvicorn app:app --host 0.0.0.0 --port 6010 &

nohup: appending output to 'nohup.out'


In [11]:
!ps -ax | grep uvicorn

   3128 ?        Rl     0:00 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 6
   3134 ?        S      0:00 /bin/bash -c ps -ax | grep uvicorn
   3136 ?        S      0:00 grep uvicorn


In [12]:
from pyngrok import ngrok

# Expose the FastAPI app
public_url = ngrok.connect(6010)
print(f"Public URL: {public_url}")

Public URL: NgrokTunnel: "https://6073-34-125-4-15.ngrok-free.app" -> "http://localhost:6010"


## Alert!

Run the following commands only at the end, to stop the ngrok and uvicorn service


In [13]:
ngrok.kill()

In [14]:
!kill -9 <pid of uvicorn service>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `kill -9 <pid of uvicorn service>'
